In [2]:
import pandas as pd
import os
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login
import json

hf_token = os.getenv("HF_TOKEN")

login(hf_token)

### Dataset Structure

In [5]:
# all datasets used in the project
DATASETS = [
    {
        "name": "WildGuardMix",
        "loader": lambda: pd.concat([
            pd.read_parquet("hf://datasets/allenai/wildguardmix/train/wildguard_train.parquet"),
            pd.read_parquet("hf://datasets/allenai/wildguardmix/test/wildguard_test.parquet"),
        ], ignore_index=True),
        "prompt_col": "prompt",
        "label_col": "prompt_harm_label",
        "malicious_value": "harmful",
        "benign_value": "unharmful",
    },
    {
        "name": "Malicious LLM Prompts",
        "loader": lambda: pd.concat([
            pd.read_parquet("hf://datasets/codesagar/malicious-llm-prompts/data/train-00000-of-00001-2279f74035821199.parquet"),
            pd.read_parquet("hf://datasets/codesagar/malicious-llm-prompts/data/validation-00000-of-00001-a3f9d7ca008d6126.parquet"),
            pd.read_parquet("hf://datasets/codesagar/malicious-llm-prompts/data/test-00000-of-00001-fec3804704adbfa8.parquet"),
        ], ignore_index=True),
        "prompt_col": "prompt",
        "label_col": "malicious",
        "malicious_value": True,
        "benign_value": False,
    },
    {
        "name": "MPDD",
        "loader": lambda: pd.read_csv("../data/MPDD.csv"),
        "prompt_col": "Prompt",
        "label_col": "isMalicious",
        "malicious_value": 0,
        "benign_value": 1,
    },
    {
        "name": "Prompt Injection & Benign Prompt Dataset",
        "loader": lambda: pd.read_json("../data/Prompt_INJECTION_And_Benign_DATASET.jsonl", lines=True),
        "prompt_col": "prompt",
        "label_col": "label",
        "malicious_value": "malicious",
        "benign_value": "benign",
    },
    {
        "name": "malicious-llm-prompts-v4",
        "loader": lambda: pd.concat([
            pd.read_parquet("hf://datasets/codesagar/malicious-llm-prompts-v4/data/train-00000-of-00001-a56d4b725b007225.parquet"),
            pd.read_parquet("hf://datasets/codesagar/malicious-llm-prompts-v4/data/test-00000-of-00001-648610b8af893e5a.parquet"),
        ], ignore_index=True),
        "prompt_col": "prompt",
        "label_col": "malicious",
        "malicious_value": True,
        "benign_value": False,
    },
    {
        "name": "deepset/prompt-injections",
        "loader": lambda: pd.concat([
            pd.read_parquet("hf://datasets/deepset/prompt-injections/data/train-00000-of-00001-9564e8b05b4757ab.parquet"),
            pd.read_parquet("hf://datasets/deepset/prompt-injections/data/test-00000-of-00001-701d16158af87368.parquet"),
        ], ignore_index=True),
        "prompt_col": "text",
        "label_col": "label",
        "malicious_value": 1,
        "benign_value": 0,
    },
    {
        "name": "jackhhao/jailbreak-classification",
        "loader": lambda: pd.concat([
            pd.read_csv("hf://datasets/jackhhao/jailbreak-classification/balanced/jailbreak_dataset_train_balanced.csv"),
            pd.read_csv("hf://datasets/jackhhao/jailbreak-classification/balanced/jailbreak_dataset_test_balanced.csv"),
        ], ignore_index=True),
        "prompt_col": "prompt",
        "label_col": "type",
        "malicious_value": "jailbreak",
        "benign_value": "benign",
    },
]

### Dataset Normalization and Merge Process

In [9]:
merged_parts = []
summary_rows = []

for ds in DATASETS:
    df = ds["loader"]().copy()

    # keep only rows with the two expected label values
    df = df[df[ds["label_col"]].isin([ds["malicious_value"], ds["benign_value"]])].copy()

    # normalize schema (malicious: 1, benign: 0)
    df["prompt"] = df[ds["prompt_col"]].astype(str).str.strip()
    df["label"] = df[ds["label_col"]].map({
        ds["malicious_value"]: 1,
        ds["benign_value"]: 0
    })
    df["source_dataset"] = ds["name"] # which dataset each row came from

    # keep only the standardized columns
    merged_parts.append(df[["prompt", "label", "source_dataset"]])

    # collect summary information about the processed dataset
    summary_rows.append({
        "dataset": ds["name"],
        "rows_kept": len(df),
        "malicious_count": int((df["label"] == 1).sum()),
        "benign_count": int((df["label"] == 0).sum()),
        "prompt_column_used": ds["prompt_col"],
        "label_column_used": ds["label_col"],
    })

In [10]:
merged_df = pd.concat(merged_parts, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)

### Per-dataset summary

In [15]:
display(summary_df)

,dataset,rows_kept,malicious_count,benign_count,prompt_column_used,label_column_used
0,WildGuardMix,88458,46970,41488,prompt,prompt_harm_label
1,Malicious LLM Prompts,5098,1758,3340,prompt,malicious
2,MPDD,39234,19617,19617,Prompt,isMalicious
3,Prompt Injection & Benign Prompt Dataset,500,250,250,prompt,label
4,malicious-llm-prompts-v4,8660,1799,6861,prompt,malicious
5,deepset/prompt-injections,662,263,399,text,label
6,jackhhao/jailbreak-classification,1306,666,640,prompt,type


### Merged dataset shape

In [19]:
print(merged_df.shape)

(143918, 3)


### Overall label distribution

In [23]:
print(merged_df["label"].value_counts().rename(index={1: "malicious", 0: "benign"}))

label
benign       72595
malicious    71323
Name: count, dtype: int64


### Sample rows

In [27]:
display(merged_df.head())

,prompt,label,source_dataset
0,Here's what I need you to do for me: 1. Transl...,0,WildGuardMix
1,I need some information quickly: 1. What is th...,0,WildGuardMix
2,Please do the following tasks: 1. Explain what...,0,WildGuardMix
3,Could you help with these items? 1. Define 'Ma...,0,WildGuardMix
4,I'm curious about a few things: 1. What's the ...,0,WildGuardMix


### Save Merged dataset

In [ ]:
merged_df.to_csv("dataset.csv", index=False)